# Experiment matrix — which branch actually helps LFM_RADAR?

Runs the E5 cells from `docs/POST_STAGE1_FIXES.md` on the **rebuilt dataset** (the one
with the RadChar measured-SNR fix), then scores them all on the same held-out test
split and prints one comparison table.

## What this answers, and what it does not

**Answers:** of `stamp_branch`, `stft_keep_rows` (C2) and `stft_freq_summary`, which
moves LFM_RADAR precision — and whether a gain there costs FHSS or JAMMING.

**Does not answer:** "is this above the 80% gate". That needs the 5-seed ensemble
(`scripts/train_ensemble.py`) plus calibration, run *after* you know which
architecture to spend 13 GPU-hours on. These cells are one model each, deliberately:
the effect being tested is roughly 5x a single model's seed spread.

## Why four cells and not just "all three on"

All three at once is one of the cells — but it cannot be the only one:

- They target **different problems**. `stft_freq_summary` and `stft_keep_rows` both
  attack the frequency axis being averaged away (jammed FHSS). `stamp_branch` is a
  matched-filter bank aimed at radar precision. Two fixes, two targets.
- Classes here **trade against each other**. `probe_jsr.py`'s docstring records
  three successive fixes where FHSS recall rose 82.5 → 89.7 → 92.2 while JAMMING
  recall fell 80.0 → 73.3 → 67.5. Stacked "improvements" are not automatically
  additive.
- **Capacity costs.** Baseline is 148,938 parameters; C2 + freq_summary is 215,437;
  the stamp branch adds more. Your last training curve already bottomed out at
  epoch 7 and rose after — this model overfits already.
- If all-three wins you still would not know **which** part won, or whether one arm
  is quietly hurting. If it loses, you learn nothing about the individual arms.

Baseline is **not optional**: every earlier number on this project was measured on the
*old* dataset, so nothing you have is comparable any more. The baseline cell is also
what tells you how big a difference has to be before it means anything.

Budget: 4 cells x ~2.5 h on a GPU. They write straight to Drive, so you can run them
across several sessions and lose nothing to a disconnect.

## 1. Code

Clones from GitHub, so the branch must be **pushed**.

In [ ]:
BRANCH = 'main'

%cd /content
!rm -rf sedicAI_NEXA
!git clone -q -b $BRANCH https://github.com/eavan127/sedicAI_NEXA.git
%cd /content/sedicAI_NEXA
!git log --oneline -1

In [ ]:
!pip install -q pyyaml h5py

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'CPU ONLY - stop and switch the runtime to GPU (Runtime > Change runtime type)')

### Check the comparison script came with the clone

`scripts/compare_experiments.py` is new. If this cell says it is missing, it has not
been pushed yet — push it, then re-run the clone cell above. (Fallback: uncomment the
`files.upload()` cell and upload it by hand.)

In [ ]:
import pathlib
p = pathlib.Path('scripts/compare_experiments.py')
print('OK, found it' if p.exists() else
      'MISSING - push scripts/compare_experiments.py, then re-run the clone cell')

In [ ]:
# Fallback only, if the script is not on the branch yet.
# from google.colab import files
# import shutil, os
# os.makedirs('scripts', exist_ok=True)
# for name in files.upload():
#     shutil.move(name, f'scripts/{name}')

## 2. Data

From `MyDrive/sedic/radar-fix-data/` — the rebuilt dataset, with real RadChar radar
bucketed by **measured** SNR rather than RadChar's own label.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATA = '/content/drive/MyDrive/sedic/radar-fix-data'
!mkdir -p data/processed
!cp $DRIVE_DATA/X.npy          data/processed/
!cp $DRIVE_DATA/y.npy          data/processed/
!cp $DRIVE_DATA/snr_labels.npy data/processed/
!ls -la data/processed/

### Verify before spending GPU hours

Checks shape, multi-hot labels, and that the SNR bins match the config — the specific
thing that was wrong once before. **If an assert fires, stop.**

In [ ]:
import numpy as np
from src.config import CFG, CLASSES

X = np.load('data/processed/X.npy', mmap_mode='r')
y = np.load('data/processed/y.npy')
snr = np.load('data/processed/snr_labels.npy')

print('X  ', X.shape, X.dtype)
print('y  ', y.shape, y.dtype)
print('snr', snr.shape)

assert X.ndim == 3 and X.shape[1] == 2, f'expected (N, 2, L), got {X.shape}'
assert y.ndim == 2 and y.shape[1] == len(CLASSES), f'y must be multi-hot (N, {len(CLASSES)}), got {y.shape}'
assert len(X) == len(y) == len(snr), 'X / y / snr lengths disagree'

bins_found, bins_cfg = sorted(set(snr.tolist())), sorted(CFG['snr_bins_db'])
print('\nSNR bins in data  :', bins_found)
print('SNR bins in config:', bins_cfg)
assert bins_found == [float(b) for b in bins_cfg], 'SNR bins do not match the config'

print('\nwindows per class (present in this many):')
for c in CLASSES:
    print(f'  {c:<12} {int(y[:, CLASSES.index(c)].sum()):,}')
print(f'\ncomposite windows (>1 class): {int((y.sum(1) > 1).sum()):,}')
print('\nOK')

### Prove the switches actually apply

Constructs each variant and checks the flags took, plus the parameter cost. Seconds,
and it catches a bad clone before a 10-hour run.

In [ ]:
from src.models.amc_cnn import AMC_CNN
from src.config import CFG, CLASSES

for name, fs, kr, st in [('baseline',        False, False, False),
                          ('stamp',           False, False, True),
                          ('rows (C2)',       False, True,  False),
                          ('all three',       True,  True,  True)]:
    m = AMC_CNN(num_classes=len(CLASSES), input_len=CFG['signal']['window_len'],
                stft_freq_summary=fs, stft_keep_rows=kr, stamp_branch=st)
    assert m.stft_branch.freq_summary == fs and m.stft_branch.keep_rows == kr
    assert (m.stamp_branch is not None) == st
    print(f"{name:<12} freq={fs!s:<5} rows={kr!s:<5} stamp={st!s:<5} "
          f"params={sum(p.numel() for p in m.parameters()):,}")
    del m

## 3. Train the four cells

Each writes `experiment_<tag>.pt` + history + a config sidecar **straight to Drive**, so
a disconnect costs you at most the cell in flight. Run them in whatever order; you can
close the tab between cells and come back.

| cell | tag | what it tests |
|---|---|---|
| baseline | `baseline` | reference + run-to-run noise |
| stamp | `stamp` | matched-filter bank, aimed at radar precision |
| rows | `rows` | C2, keeps the frequency rows |
| all three | `freq_rows_stamp` | your "do they work together" question |

In [ ]:
OUT = '/content/drive/MyDrive/sedic/experiments'
!mkdir -p $OUT
print(OUT)

In [ ]:
!python scripts/run_stft_experiment.py --no-freq-summary --out-dir $OUT

In [ ]:
!python scripts/run_stft_experiment.py --no-freq-summary --stamp-branch --out-dir $OUT

In [ ]:
!python scripts/run_stft_experiment.py --no-freq-summary --keep-rows --out-dir $OUT

In [ ]:
!python scripts/run_stft_experiment.py --keep-rows --stamp-branch --out-dir $OUT

## 4. Compare

Scores every checkpoint in `$OUT` on the same held-out test split.

**Read the AP column first.** Average precision is computed over the raw sigmoid
scores at every threshold, so it measures whether a variant genuinely separates the
classes better. `P@0.5` uses the flat fallback, **not** the calibrated per-class
thresholds in the config (LFM_RADAR sits at 0.22 there) — those were fitted to an old
ensemble to buy recall margin, so reading precision through them measures the
calibration, not the model.

A variant only wins if it beats baseline by more than baseline's own run-to-run
spread.

In [ ]:
!python scripts/compare_experiments.py --results-dir $OUT --out $OUT/experiment_comparison.json

### Optional: where the radar misses go

`high_snr_probe.py` splits recall by standalone / mixture / overlay and reports what
the model says *instead* when it misses — this is where a radar-vs-FHSS or
radar-vs-barrage confusion shows up directly. Flags must match the checkpoint or it
fails loudly.

In [ ]:
# !python scripts/high_snr_probe.py --n 300 --class LFM_RADAR \
#     --checkpoint $OUT/experiment_stamp.pt --stamp-branch

## 5. After this

Everything is already on Drive (`sedic/experiments/`). Nothing to rescue.

Next, **locally**:

1. Pick the winner from the AP table — and only if it clears baseline by more than
   the noise floor.
2. Retrain that one as the full 5-seed ensemble (`scripts/train_ensemble.py --models 5`)
   — one model tells you which architecture, not whether it passes the gate.
3. `python scripts/calibrate_thresholds.py --ensemble --n-models 5`, paste the values
   into `configs/default.yaml`.
4. `python -m src.evaluate` for the final scorecard.

Leave `stft_freq_summary`, `stft_keep_rows` and `stamp_branch` **false** in
`configs/default.yaml` throughout. With any of them true on disk, the fused tensor
shape changes and none of the shipped checkpoints load — the experiment runner flips
them in memory precisely to avoid leaving that landmine behind. The winning flag only
goes into the config when you retrain the ensemble to match, and the old checkpoints
are retired in the same step.